# Notebook 00 — Environment and Data Download

**Task ID:** `ENV-001`  
**Phase:** Phase 0 — Environment Setup & Data Ingestion  
**Purpose:** Establish a reproducible Colab / Local environment, configure dependencies, and download all datasets without exposing credentials.

---

## 1. Project Objective and Dataset List

### Datasets Used in ClaimVision AI:
1. **Fraud Dataset:** `vinayjose/car-damage-dataset` (Kaggle)
   - Used for visual fraud-risk classification (Phase 1 & 2).
2. **Severity Dataset:** `anujms/car-damage-severity-dataset` (Kaggle)
   - 3-class damage severity: Minor, Moderate, Severe (Phase 5–9).
3. **Detection Dataset:** COCO Car Damage Detection Dataset
   - Damage localization and 5 damaged-part classes (Phase 10–12).

## 2. Universal Colab & Local Environment Bootstrap

In [ ]:
import os
import sys
import random
from pathlib import Path

# --- Colab / Local Universal Sync (Clone if new, Pull if already exists) ---
if 'google.colab' in sys.modules or os.path.exists('/content'):
    repo_dir = Path('/content/NPN-Car-Insurance')
    import subprocess
    if not (repo_dir / '.git').exists():
        print('🚀 New Colab session detected. Cloning repository...')
        subprocess.run(['git', 'clone', 'https://github.com/AmitavaDatta2004/NPN-Car-Insurance.git', str(repo_dir)], check=True)
    else:
        print('🔄 Colab repository already exists. Pulling latest commits from GitHub...')
        subprocess.run(['git', '-C', str(repo_dir), 'pull'], check=True)
    
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'imagehash', 'kagglehub'], check=False)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo_dir / 'ml')], check=False)
    
    ml_src = str(repo_dir / 'ml' / 'src')
    if ml_src not in sys.path:
        sys.path.insert(0, ml_src)
    os.chdir(str(repo_dir))
    print('✅ Working directory updated to:', os.getcwd())
else:
    # Local fallback for running directly on your laptop
    for candidate in [Path('ml/src').resolve(), Path('../ml/src').resolve(), Path('../../ml/src').resolve()]:
        if candidate.exists() and str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
            break



## 3. GPU and System Hardware Verification

In [ ]:
import platform
import torch

print('=' * 50)
print(f'OS / Platform      : {platform.platform()}')
print(f'Python Version     : {platform.python_version()}')
print(f'PyTorch Version    : {torch.__version__}')
print(f'CUDA Available     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device Name    : {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory (VRAM)  : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
else:
    print('GPU Device Name    : CPU (No CUDA device found)')
print('=' * 50)


## 4. Reproducible Random Seed Initialization

In [ ]:
import numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
    print(f'🔒 Global scientific reproducibility seed locked to: {seed}')

set_seed(42)


## 5. Download Fraud Dataset (`vinayjose/car-damage-dataset`)

In [ ]:
import kagglehub

raw_dir = Path('data/raw').resolve()
raw_dir.mkdir(parents=True, exist_ok=True)
os.environ['KAGGLEHUB_CACHE'] = str(raw_dir)

fraud_dir = raw_dir / 'vinayjose_car_damage'

# Check if dataset is already downloaded and non-empty to avoid re-downloading
if fraud_dir.exists() and any(fraud_dir.iterdir()):
    print(f'✅ Fraud dataset already present at: {fraud_dir} (Skipping download)')
    fraud_path = str(fraud_dir)
else:
    print(f'Downloading fraud dataset to: {fraud_dir}...')
    try:
        fraud_path = kagglehub.dataset_download('vinayjose/car-damage-dataset', output_dir=str(fraud_dir))
        print(f'✅ Fraud dataset successfully downloaded to: {fraud_path}')
    except Exception as e:
        print(f'⚠️ Kagglehub download note: {e}')
        fraud_path = str(fraud_dir)


## 6. Download Severity Dataset (`anujms/car-damage-severity-dataset`)

In [ ]:
severity_dir = raw_dir / 'car_damage_severity'

# Check if dataset is already downloaded and non-empty to avoid re-downloading
if severity_dir.exists() and any(severity_dir.iterdir()):
    print(f'✅ Severity dataset already present at: {severity_dir} (Skipping download)')
    sev_path = str(severity_dir)
else:
    print(f'Downloading severity dataset to: {severity_dir}...')
    try:
        sev_path = kagglehub.dataset_download('anujms/car-damage-severity-dataset', output_dir=str(severity_dir))
        print(f'✅ Severity dataset successfully downloaded to: {sev_path}')
    except Exception as e:
        print(f'⚠️ Kagglehub download note: {e}')
        sev_path = str(severity_dir)


## 7. Dataset Directory Structure & Registry Creation

In [ ]:
import json
import hashlib

registry = {
    'fraud_dataset': {
        'source': 'vinayjose/car-damage-dataset',
        'path': str(fraud_dir),
        'exists': fraud_dir.exists(),
    },
    'severity_dataset': {
        'source': 'anujms/car-damage-severity-dataset',
        'path': str(severity_dir),
        'exists': severity_dir.exists(),
    }
}

registry_path = Path('data/dataset_registry.json')
registry_path.parent.mkdir(parents=True, exist_ok=True)
with open(registry_path, 'w') as f:
    json.dump(registry, f, indent=2)

print(f'✅ Dataset registry saved to: {registry_path}')
print(json.dumps(registry, indent=2))
